# 🎵 Music Genre Classification — Enhanced AST Fine-Tuning
### Target: 0.87+ Macro F1
**Architecture:** Audio Spectrogram Transformer (AST) — `MIT/ast-finetuned-audioset-10-10-0.4593`

**Improvements over baseline AST notebook:**
- 3-phase training (Head → Top-6 transformer layers → Full fine-tune)
- Pitch shift augmentation (±3 semitones)
- SpecAugment on log-mel tensors (frequency + time masking)
- Mixup at batch level (50% of batches)
- Focal Loss (γ=2) — focuses on hard/confused genres
- Mixed-precision training (FP16 with GradScaler)
- DURATION = 25 s (+25% more audio context)
- TRAIN_SIZE = 7000 (+40% more synthetic diversity)
- N_TTA = 8 (more crops, lower inference variance)
- LR_P3 = 1e-5 (safer full fine-tune)
- Rich 6-section EDA (waveforms, MFCCs, spectral heatmap, augmentation showcase)

---
### ⚙️ Key Parameters
| Parameter | Default | Effect |
|-----------|---------|--------|
| `DURATION` | 25 s | More audio context |
| `TRAIN_SIZE` | 7000 | More augmented diversity |
| `EPOCHS_P3` | 6 | Full fine-tune depth |
| `LR_P3` | 1e-5 | Safe — preserves AudioSet features |
| `N_TTA` | 8 | Lower prediction variance |
| `MIXUP_ALPHA` | 0.4 | Interpolation strength |
| `PITCH_PROB` | 0.3 | Pitch invariance robustness |


## Cell 1 — Install Libraries

In [ ]:
# ── CELL 1: Install libraries ──────────────────────────
!pip install -q transformers librosa torchaudio torchmetrics wandb matplotlib seaborn scikit-learn


## Cell 2 — Imports

In [ ]:
# ── CELL 2: Imports ────────────────────────────────────
import os, random, warnings, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio                             # fast audio I/O + pitch shift
import torchaudio.functional as TA
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    get_cosine_schedule_with_warmup,
)
from torchmetrics.classification import MulticlassF1Score
from sklearn.metrics import confusion_matrix, classification_report
warnings.filterwarnings('ignore')
print('All imports done!')


## Cell 3 — Seed & Device

In [ ]:
# ── CELL 3: Seed for reproducibility ──────────────────
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## Cell 4A — Kaggle CONFIG

Run on Kaggle. Comment out Cell 4A and run Cell 4B when working locally.

In [ ]:
# ── CELL 4A: CONFIG — Kaggle environment ───────────────
SAMPLE_RATE  = 16000
DURATION     = 25                         # 25 s — 25% more context than baseline
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 6                          # keep small — AST is memory-heavy
GRAD_ACCUM   = 2                          # effective batch = 12
NUM_WORKERS  = 2

EPOCHS_P1    = 3                          # Phase 1: classifier head only
EPOCHS_P2    = 3                          # Phase 2: top-6 encoder layers + head
EPOCHS_P3    = 6                          # Phase 3: full fine-tune
LR_P1        = 3e-4                       # high OK — base frozen
LR_P2_HEAD   = 1e-4                       # Phase 2 head LR
LR_P2_TOP    = 5e-5                       # Phase 2 top-6 layers LR
LR_P3        = 1e-5                       # Phase 3 base LR (KEEP TINY!)
LR_P3_HEAD   = 1e-4                       # Phase 3 head LR (10×)
WEIGHT_DECAY = 0.01
PATIENCE_P1  = 2
PATIENCE_P2  = 2
PATIENCE_P3  = 3

TRAIN_SIZE   = 7000                       # 40% more synthetic samples
NOISE_PROB   = 0.70
NOISE_LEVEL  = (0.03, 0.20)
TEMPO_PROB   = 0.35
TEMPO_RANGE  = (0.88, 1.12)
PITCH_PROB   = 0.30                       # NEW: pitch shift robustness
PITCH_RANGE  = (-3.0, 3.0)               # semitones

MIXUP_ALPHA  = 0.4                        # NEW: Mixup beta distribution alpha
MIXUP_PROB   = 0.50                       # fraction of batches that get mixed

# SpecAugment parameters (applied to log-mel tensor)
FREQ_MASK_F  = 20                         # max frequency bins to mask
TIME_MASK_T  = 40                         # max time frames to mask
NUM_MASKS    = 2                          # number of masks per axis

N_TTA        = 8                          # TTA crops (was 5 in baseline)

BASE_PATH    = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
OUTPUT_PATH  = '/kaggle/working'
USE_WANDB    = True

print('Kaggle CONFIG loaded!')
print(f'  DURATION={DURATION}s | TRAIN_SIZE={TRAIN_SIZE} | N_TTA={N_TTA}')


## Cell 4B — Local CONFIG

Run locally. Comment out Cell 4B and run Cell 4A on Kaggle.

In [ ]:
# ── CELL 4B: CONFIG — Local / offline environment ──────
SAMPLE_RATE  = 16000
DURATION     = 25
MAX_LENGTH   = SAMPLE_RATE * DURATION

BATCH_SIZE   = 4
GRAD_ACCUM   = 2
NUM_WORKERS  = 0                          # 0 on Windows to avoid spawn deadlock

EPOCHS_P1    = 3
EPOCHS_P2    = 3
EPOCHS_P3    = 6
LR_P1        = 3e-4
LR_P2_HEAD   = 1e-4
LR_P2_TOP    = 5e-5
LR_P3        = 1e-5
LR_P3_HEAD   = 1e-4
WEIGHT_DECAY = 0.01
PATIENCE_P1  = 2
PATIENCE_P2  = 2
PATIENCE_P3  = 3

TRAIN_SIZE   = 2000
NOISE_PROB   = 0.70
NOISE_LEVEL  = (0.03, 0.20)
TEMPO_PROB   = 0.35
TEMPO_RANGE  = (0.88, 1.12)
PITCH_PROB   = 0.30
PITCH_RANGE  = (-3.0, 3.0)

MIXUP_ALPHA  = 0.4
MIXUP_PROB   = 0.50

FREQ_MASK_F  = 20
TIME_MASK_T  = 40
NUM_MASKS    = 2

N_TTA        = 3

BASE_PATH    = 'F:/Downloads/jan-2026-dl-gen-ai-project/messy_mashup'
OUTPUT_PATH  = 'F:/Downloads/jan-2026-dl-gen-ai-project/Models'
USE_WANDB    = False

print('Local CONFIG loaded!')
print(f'  DURATION={DURATION}s | TRAIN_SIZE={TRAIN_SIZE} | N_TTA={N_TTA}')


## Cell 5 — Paths & Labels

In [ ]:
# ── CELL 5: Paths & Labels ─────────────────────────────
GENRES_PATH    = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH       = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
REQUIRED_STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES   = ['blues', 'classical', 'country', 'disco', 'hiphop',
            'jazz', 'metal', 'pop', 'reggae', 'rock']
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

if not os.path.exists(GENRES_PATH):
    raise FileNotFoundError(f'Data path not found: {GENRES_PATH}')
print('Paths set!')
print('label2id:', label2id)


## Cell 6 — Build Song Index

In [ ]:
# ── CELL 6: Build Song Index ───────────────────────────
def build_song_index() -> dict:
    index = {}
    for genre in GENRES:
        genre_path = os.path.join(GENRES_PATH, genre)
        valid = [
            os.path.join(genre_path, folder)
            for folder in sorted(os.listdir(genre_path))
            if os.path.isdir(os.path.join(genre_path, folder))
            and all(
                os.path.exists(os.path.join(genre_path, folder, s))
                for s in REQUIRED_STEMS
            )
        ]
        index[genre] = valid
        print(f'  {genre}: {len(valid)} songs')
    return index

song_index = build_song_index()

noise_files = [
    os.path.join(root, f)
    for root, _, files in os.walk(ESC_PATH)
    for f in files if f.endswith('.wav')
]
print(f'\nNoise files: {len(noise_files)}')


## Cell 7 — Train / Validation Split (85 / 15)

In [ ]:
# ── CELL 7: Train / Val Split ──────────────────────────
train_index, val_index = {}, {}

for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    split = int(0.85 * len(songs))
    train_index[genre] = songs[:split]
    val_index[genre]   = songs[split:]
    print(f'  {genre} -> Train: {len(train_index[genre])}, Val: {len(val_index[genre])}')

print('\nTrain/Val split done!')


## Cell 8 — Audio Helper Functions

All worker-path ops now use **only torchaudio + numpy** — no librosa, no scipy, no numba.
librosa is kept exclusively for EDA cells (run in the main process, no workers involved).

| Function | Library | Reason |
|----------|---------|--------|
| `load_audio` | torchaudio | 2-3× faster WAV I/O, no scipy/numba |
| `tempo_augment` | torchaudio resample | Speed-change via resampling — avoids numba STFT deadlock in workers |
| `pitch_shift_augment` | torchaudio | 3-5× faster than librosa, no scipy phase vocoder |
| EDA spectrograms | librosa | Main-process only, no worker risk |

**Root cause of the 0% hang:** `librosa.effects.time_stretch` calls scipy STFT → triggers numba JIT → numba tries to acquire a CUDA context already locked by the main process → workers deadlock before producing a single sample.


In [ ]:
# ── CELL 8: Audio Helpers ──────────────────────────────
# ALL worker-path functions use only torchaudio + numpy.
# librosa is NEVER called inside __getitem__ (causes numba/CUDA deadlock in workers).

def load_audio(path: str) -> np.ndarray:
    """torchaudio I/O — 2-3× faster than librosa.load, no scipy/numba."""
    waveform, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        waveform = TA.resample(waveform, sr, SAMPLE_RATE)
    return waveform.mean(0).numpy().astype(np.float32)

def normalize(audio: np.ndarray) -> np.ndarray:
    return audio / (np.max(np.abs(audio)) + 1e-6)

def crop_random(audio: np.ndarray) -> np.ndarray:
    if len(audio) >= MAX_LENGTH:
        start = random.randint(0, len(audio) - MAX_LENGTH)
        return audio[start: start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def crop_center(audio: np.ndarray) -> np.ndarray:
    if len(audio) >= MAX_LENGTH:
        start = (len(audio) - MAX_LENGTH) // 2
        return audio[start: start + MAX_LENGTH]
    return np.pad(audio, (0, MAX_LENGTH - len(audio)))

def random_gain(audio: np.ndarray) -> np.ndarray:
    return audio * random.uniform(0.7, 1.3)

def tempo_augment(audio: np.ndarray) -> np.ndarray:
    """Speed change via torchaudio resampling — no numba, no scipy, no deadlock.

    Treats the audio as if it was recorded at fake_sr = SR * rate, then
    resamples back to SR. This stretches/compresses time by 1/rate with a very
    small pitch shift (±6% for ±12% tempo range) — acceptable for genre augmentation,
    and a blues song sped up slightly is still blues.

    Replaces librosa.effects.time_stretch which triggered numba JIT on first call
    inside a worker, deadlocking on the CUDA context held by the main process.
    """
    if random.random() < TEMPO_PROB:
        rate   = random.uniform(*TEMPO_RANGE)
        tensor = torch.from_numpy(audio).unsqueeze(0)              # (1, T)
        tensor = TA.resample(tensor, int(SAMPLE_RATE * rate), SAMPLE_RATE)
        return tensor.squeeze(0).numpy()
    return audio

def pitch_shift_augment(audio: np.ndarray) -> np.ndarray:
    """torchaudio pitch shift — 3-5× faster than librosa, no scipy phase vocoder."""
    if random.random() < PITCH_PROB:
        n_steps = random.uniform(*PITCH_RANGE)
        tensor  = torch.from_numpy(audio).unsqueeze(0)
        tensor  = TA.pitch_shift(tensor, SAMPLE_RATE, n_steps)
        return tensor.squeeze(0).numpy()
    return audio

def add_noise(audio: np.ndarray) -> np.ndarray:
    if random.random() < NOISE_PROB and noise_files:
        noise = load_audio(random.choice(noise_files))
        noise = crop_random(noise)
        level = random.uniform(*NOISE_LEVEL)
        audio = audio + level * noise
    return audio

print('Audio helpers ready! (all worker-path ops: torchaudio only)')
print('  load_audio    : torchaudio.load + TA.resample')
print('  tempo_augment : torchaudio resample trick  (no numba/scipy)')
print('  pitch_shift   : TA.pitch_shift             (no numba/scipy)')
print('  add_noise     : torchaudio.load + numpy')


## Cell 9 — WandB Initialisation

In [ ]:
# ── CELL 9: WandB Init ─────────────────────────────────
if USE_WANDB:
    os.environ['WANDB_DISABLE_SERVICE'] = 'true'
    try:
        from kaggle_secrets import UserSecretsClient
        import wandb
        wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
    except Exception:
        import wandb
        wandb.login()

    wandb_run = wandb.init(
        project = '22f3003226-t12026',
        name    = 'ast-enhanced-25s-focal-mixup',
        mode    = 'offline',
        config  = dict(
            model        = 'MIT/ast-finetuned-audioset-10-10-0.4593',
            duration     = DURATION,
            train_size   = TRAIN_SIZE,
            batch_size   = BATCH_SIZE,
            n_tta        = N_TTA,
            mixup_alpha  = MIXUP_ALPHA,
            pitch_prob   = PITCH_PROB,
            focal_gamma  = 2.0,
        ),
    )
    print(f'WandB run: {wandb_run.name}')
else:
    wandb = None
    print('WandB disabled.')


---
## 📊 Exploratory Data Analysis

Six sections, each revealing something distinct about the dataset and the acoustic differences between genres.

1. **Genre & split distribution** — verifying balance
2. **Audio duration** — how much the 25 s crop covers
3. **Waveforms with RMS envelope** — visual genre fingerprints
4. **Log-mel spectrogram grid** — what the AST actually sees
5. **MFCC heatmap across genres** — timbre fingerprint comparison
6. **Augmentation showcase** — what the training dataset looks like


## Cell 10 — EDA: Genre & Split Distribution

In [ ]:
# ── CELL 10: EDA — Genre Distribution ─────────────────
import matplotlib.pyplot as plt
import seaborn as sns

train_counts = {g: len(train_index[g]) for g in GENRES}
val_counts   = {g: len(val_index[g])   for g in GENRES}
total_counts = {g: len(song_index[g])  for g in GENRES}

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
palette = sns.color_palette('Set2', len(GENRES))

for ax, counts, title in zip(
    axes,
    [total_counts, train_counts, val_counts],
    ['Total Songs per Genre', 'Train Split', 'Validation Split'],
):
    bars = ax.bar(counts.keys(), counts.values(), color=palette)
    for bar, v in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                str(v), ha='center', va='bottom', fontsize=8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Genre')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Dataset Distribution', fontsize=15, y=1.02)
plt.tight_layout()
if USE_WANDB: wandb.log({'eda/genre_distribution': wandb.Image(fig)})
plt.show(); plt.close(fig)

print(f'Total songs : {sum(total_counts.values())}')
print(f'Train songs : {sum(train_counts.values())}')
print(f'Val songs   : {sum(val_counts.values())}')


## Cell 11 — EDA: Audio Duration Distribution

Sample 10 songs per genre (drum stem) and measure actual clip lengths. The red dashed line shows our 25 s crop — we want it to cover the majority of songs without too much padding.

In [ ]:
# ── CELL 11: EDA — Duration Distribution ──────────────
durations_by_genre = {}
for genre in GENRES:
    durs = []
    for sp in train_index[genre][:10]:
        try:
            a = load_audio(os.path.join(sp, 'drums.wav'))
            durs.append(len(a) / SAMPLE_RATE)
        except Exception:
            pass
    durations_by_genre[genre] = durs

all_durs = [d for durs in durations_by_genre.values() for d in durs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Global histogram
ax1.hist(all_durs, bins=25, color='steelblue', edgecolor='white')
ax1.axvline(DURATION, color='red', linestyle='--', linewidth=2,
            label=f'Crop = {DURATION}s')
ax1.set_title('Duration Histogram (all genres, drum stem)')
ax1.set_xlabel('Duration (s)'); ax1.set_ylabel('Count')
ax1.legend()

# Boxplot per genre
ax2.boxplot(
    [durations_by_genre[g] for g in GENRES],
    labels=GENRES, patch_artist=True,
    boxprops=dict(facecolor='steelblue', alpha=0.6),
)
ax2.axhline(DURATION, color='red', linestyle='--', linewidth=1.5,
            label=f'Crop = {DURATION}s')
ax2.set_title('Duration Boxplot per Genre')
ax2.set_xlabel('Genre'); ax2.set_ylabel('Duration (s)')
ax2.tick_params(axis='x', rotation=30)
ax2.legend()

plt.tight_layout()
if USE_WANDB: wandb.log({'eda/duration_distribution': wandb.Image(fig)})
plt.show(); plt.close(fig)

pct_covered = np.mean([d <= DURATION for d in all_durs]) * 100
print(f'Songs fully covered by {DURATION}s crop : {pct_covered:.1f}%')


## Cell 12 — EDA: Waveforms with RMS Envelope

In [ ]:
# ── CELL 12: EDA — Waveforms ───────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    sp = train_index[genre][0]
    mixed = None
    for stem in REQUIRED_STEMS:
        a     = load_audio(os.path.join(sp, stem))
        a     = crop_center(a)
        mixed = a if mixed is None else mixed + a
    mixed = normalize(mixed)

    t = np.linspace(0, DURATION, len(mixed))
    frame = SAMPLE_RATE // 10
    rms = np.array([
        np.sqrt(np.mean(mixed[j: j + frame] ** 2))
        for j in range(0, len(mixed) - frame, frame // 2)
    ])
    t_rms = np.linspace(0, DURATION, len(rms))

    axes[i].plot(t, mixed, linewidth=0.3, color='steelblue', alpha=0.55)
    axes[i].fill_between(t_rms, -rms, rms, alpha=0.35, color='darkorange', label='RMS')
    axes[i].set_title(genre.capitalize(), fontweight='bold', fontsize=11)
    axes[i].set_xlim(0, DURATION)
    axes[i].set_xlabel('Time (s)', fontsize=8)
    axes[i].set_yticks([])

plt.suptitle('Mixed Waveforms with RMS Envelope — One Song per Genre', fontsize=14, y=1.01)
plt.tight_layout()
if USE_WANDB: wandb.log({'eda/waveforms': wandb.Image(fig)})
plt.show(); plt.close(fig)


## Cell 13 — EDA: Log-Mel Spectrogram Grid

This is exactly what the AST model processes after the feature extractor. Each cell is a 2D heatmap of frequency (mel scale) vs time — the 'image' the transformer patches over.

In [ ]:
# ── CELL 13: EDA — Log-Mel Spectrogram Grid ────────────
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    sp = train_index[genre][0]
    mixed = None
    for stem in REQUIRED_STEMS:
        a     = load_audio(os.path.join(sp, stem))
        a     = crop_center(a)
        mixed = a if mixed is None else mixed + a
    mixed = normalize(mixed)

    mel    = librosa.feature.melspectrogram(
        y=mixed, sr=SAMPLE_RATE, n_mels=128, fmax=8000
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    librosa.display.specshow(
        mel_db, sr=SAMPLE_RATE, x_axis='time', y_axis='mel',
        fmax=8000, ax=axes[i], cmap='magma',
    )
    axes[i].set_title(genre.capitalize(), fontweight='bold', fontsize=11)
    if i % 5 != 0:
        axes[i].set_ylabel('')

plt.suptitle('Log-Mel Spectrograms — Input to AST Feature Extractor', fontsize=14, y=1.01)
plt.tight_layout()
if USE_WANDB: wandb.log({'eda/log_mel_grid': wandb.Image(fig)})
plt.show(); plt.close(fig)
print('AST input shape: (1024 time patches x 128 mel bins) -> 16x16 patch tokens')


## Cell 14 — EDA: MFCC Feature Heatmap

Averaged first 13 MFCCs per genre. MFCCs capture timbral texture — the unique 'sound fingerprint' that distinguishes genres. High contrast between rows indicates genres that are acoustically very different (easy to classify). Similar rows are where the model struggles.

In [ ]:
# ── CELL 14: EDA — MFCC Heatmap ───────────────────────
N_MFCC = 13
mfcc_matrix = np.zeros((len(GENRES), N_MFCC))

for i, genre in enumerate(GENRES):
    coeffs = []
    for sp in train_index[genre][:6]:
        try:
            audio = load_audio(os.path.join(sp, 'vocals.wav'))
            audio = crop_center(audio)
            mfcc  = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
            coeffs.append(np.mean(mfcc, axis=1))
        except Exception:
            pass
    if coeffs:
        mfcc_matrix[i] = np.mean(coeffs, axis=0)

# Z-score normalise per MFCC coefficient for fair comparison
mfcc_norm = (mfcc_matrix - mfcc_matrix.mean(0)) / (mfcc_matrix.std(0) + 1e-8)
mfcc_df   = pd.DataFrame(
    mfcc_norm,
    index   = [g.capitalize() for g in GENRES],
    columns = [f'MFCC-{k+1}' for k in range(N_MFCC)],
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(
    mfcc_df, annot=True, fmt='.2f', cmap='RdYlGn',
    linewidths=0.4, ax=ax, cbar_kws={'label': 'Z-score'},
)
ax.set_title('Average MFCC Coefficients per Genre (vocals stem, Z-normalised)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('MFCC Coefficient')
ax.set_ylabel('Genre')
plt.tight_layout()
if USE_WANDB: wandb.log({'eda/mfcc_heatmap': wandb.Image(fig)})
plt.show(); plt.close(fig)

print('Interpretation:')
print('  Green = high value relative to dataset mean')
print('  Red   = low value relative to dataset mean')
print('  Rows with similar patterns = genres model may confuse (e.g. blues <-> jazz)')


## Cell 15 — EDA: Augmentation Showcase

Visualise how each augmentation transforms the same audio. The AST feature extractor converts each variant to a log-mel spectrogram — this is what the model's patch embedding actually tokenises.

In [ ]:
# ── CELL 15: EDA — Augmentation Showcase ──────────────
def quick_mel_db(audio: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(y=audio, sr=SAMPLE_RATE, n_mels=128, fmax=8000)
    return librosa.power_to_db(mel, ref=np.max)

sp = train_index['rock'][1]
raw_stems = []
for stem in REQUIRED_STEMS:
    raw_stems.append(load_audio(os.path.join(sp, stem)))
base_audio = normalize(sum(crop_center(a) for a in raw_stems))

random.seed(7); np.random.seed(7)
aug_examples = {
    'Original (centre crop)':      base_audio,
    'Random crop':                 normalize(crop_random(base_audio)),
    'Tempo ×0.88':                 normalize(librosa.effects.time_stretch(base_audio, rate=0.88)),
    'Pitch +2.5 semitones':        normalize(librosa.effects.pitch_shift(base_audio, sr=SAMPLE_RATE, n_steps=2.5)),
    'ESC-50 noise (12%)':          normalize(base_audio + 0.12 * normalize(load_audio(noise_files[3]))[:MAX_LENGTH]),
}

fig, axes = plt.subplots(1, len(aug_examples), figsize=(22, 4))
for ax, (title, audio) in zip(axes, aug_examples.items()):
    mel_db = quick_mel_db(audio)
    ax.imshow(mel_db, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Augmentation Showcase — Rock Sample (each is a different augmented view)',
             fontsize=13, y=1.05)
plt.tight_layout()
if USE_WANDB: wandb.log({'eda/augmentation_showcase': wandb.Image(fig)})
plt.show(); plt.close(fig)
print('All variants preserve genre identity while creating acoustic diversity.')


---
## 🤖 Model: Audio Spectrogram Transformer (AST)

**Backbone:** `MIT/ast-finetuned-audioset-10-10-0.4593`
- 86M parameters
- Pretrained on AudioSet (2M+ YouTube clips, 527 classes)
- 12-layer transformer encoder, 768 hidden dim, 12 attention heads
- Input: log-mel spectrogram patches (16×16 tokens)

**New pipeline (3-phase):**
- Phase 1 — Classifier head only (base fully frozen)
- Phase 2 — Top 6 transformer encoder layers + head (early layers frozen)
- Phase 3 — Full fine-tune, all layers, tiny LR


## Cell 16 — Load AST Feature Extractor & Model

In [ ]:
# ── CELL 16: Load AST Feature Extractor & Model ────────
MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'

feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_NAME)
print('Feature extractor loaded!')

model = ASTForAudioClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True,   # replaces 527-class head with 10-class
)
model = model.to(DEVICE)

total = sum(p.numel() for p in model.parameters())
print(f'AST loaded on {DEVICE}')
print(f'Total parameters : {total:,}')
print(f'Encoder layers   : {len(model.audio_spectrogram_transformer.encoder.layer)}')


## Cell 17 — SpecAugment on Log-Mel Tensors

SpecAugment masks rectangular blocks of the spectrogram tensor **after** the feature extractor converts audio to `input_values`. This directly augments what the transformer sees — proven to significantly improve speech/audio models.

In [ ]:
# ── CELL 17: SpecAugment helper ────────────────────────
def spec_augment(input_values: torch.Tensor) -> torch.Tensor:
    """Apply SpecAugment to a batch of log-mel tensors.

    Args:
        input_values: shape (B, time_frames, mel_bins) — AST format

    Returns:
        Augmented tensor, same shape.
    """
    x = input_values.clone()
    B, T, F = x.shape

    for _ in range(NUM_MASKS):
        # Frequency masking
        f  = random.randint(1, min(FREQ_MASK_F, F - 1))
        f0 = random.randint(0, F - f)
        x[:, :, f0: f0 + f] = 0.0

        # Time masking
        t  = random.randint(1, min(TIME_MASK_T, T - 1))
        t0 = random.randint(0, T - t)
        x[:, t0: t0 + t, :] = 0.0

    return x

print(f'SpecAugment ready! ({NUM_MASKS} masks per axis, freq={FREQ_MASK_F}, time={TIME_MASK_T})')


## Cell 18 — Training Dataset

Adds **pitch shift** on top of the baseline augmentation pipeline. Everything else (cross-song mixing, tempo, gain, noise) is preserved from the original.

In [ ]:
# ── CELL 18: TrainDataset ──────────────────────────────
class TrainDataset(Dataset):
    """Generates TRAIN_SIZE synthetic mashups per epoch.

    Augmentation pipeline:
      1. Cross-song stem mixing (each stem from a different song)
      2. Tempo stretch  (35% chance)
      3. Pitch shift    (30% chance, ±3 semitones)  ← NEW
      4. Random gain    (always, ±30%)
      5. ESC-50 noise   (70% chance, 3-20% level)
      6. Normalize
      7. AST feature extraction -> input_values
    SpecAugment is applied inside train_one_epoch at the batch level.
    """

    def __init__(self, song_index: dict, size: int = TRAIN_SIZE):
        self.song_index = song_index
        self.size       = size

    def __len__(self) -> int:
        return self.size

    def __getitem__(self, idx: int):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if not songs:
                    continue

                mixed = None
                for stem in REQUIRED_STEMS:
                    sp    = random.choice(songs)
                    audio = load_audio(os.path.join(sp, stem))
                    audio = crop_random(audio)
                    audio = tempo_augment(audio)
                    audio = pitch_shift_augment(audio)   # NEW
                    audio = random_gain(audio)
                    mixed = audio if mixed is None else mixed + audio

                mixed = normalize(mixed)
                mixed = add_noise(mixed)
                mixed = normalize(mixed)

                inputs = feature_extractor(
                    mixed,
                    sampling_rate  = SAMPLE_RATE,
                    return_tensors = 'pt',
                )
                return {
                    'input_values': inputs['input_values'].squeeze(0),
                    'labels': torch.tensor(label2id[genre], dtype=torch.long),
                }
            except Exception:
                continue

print('TrainDataset defined! (cross-stem + tempo + pitch + gain + noise)')


## Cell 19 — Validation Dataset

Fixed centre crops, no augmentation — reproducible evaluation. Multi-crop TTA happens only at inference time.

In [ ]:
# ── CELL 19: ValDataset ────────────────────────────────
class ValDataset(Dataset):
    """Fixed centre-crop dataset. No augmentation."""

    def __init__(self, song_index: dict):
        self.samples = [
            (genre, sp)
            for genre in GENRES
            for sp in song_index[genre]
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        genre, sp = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = load_audio(os.path.join(sp, stem))
            audio = crop_center(audio)
            mixed = audio if mixed is None else mixed + audio
        mixed = normalize(mixed)
        inputs = feature_extractor(
            mixed,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = 'pt',
        )
        return {
            'input_values': inputs['input_values'].squeeze(0),
            'labels': torch.tensor(label2id[genre], dtype=torch.long),
        }

print('ValDataset defined!')


## Cell 20 — DataLoaders

In [ ]:
# ── CELL 20: DataLoaders ───────────────────────────────
_pin  = (DEVICE == 'cuda')
_pers = (NUM_WORKERS > 0)
# 'spawn' re-initialises each worker cleanly instead of forking.
# Fork is the Linux default and deadlocks when scipy/librosa thread pools
# are copied mid-state into child processes → tqdm stuck at 0%.
_ctx  = 'spawn' if NUM_WORKERS > 0 else None

train_dataset = TrainDataset(train_index, size=TRAIN_SIZE)
val_dataset   = ValDataset(val_index)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=_pin, persistent_workers=_pers,
    multiprocessing_context=_ctx,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=_pin, persistent_workers=_pers,
    multiprocessing_context=_ctx,
)

print(f'Train samples  : {len(train_dataset)}')
print(f'Val samples    : {len(val_dataset)}')
print(f'Train batches  : {len(train_loader)}')
print(f'Val batches    : {len(val_loader)}')
print(f'Worker context : {_ctx or "main process (no workers)"}')


## Cell 21 — Focal Loss + Metric + Scaler

Focal Loss is critical for this dataset because genres like blues/jazz and disco/pop have significant spectral overlap — the model's confident wrong predictions are expensive. Focal Loss penalises these hard examples more than CrossEntropy does.

In [ ]:
# ── CELL 21: Focal Loss + Metric + AMP Scaler ─────────
class FocalLoss(nn.Module):
    """Multi-class Focal Loss with label smoothing.

    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)

    gamma=2: strongly down-weights easy examples.
    label_smoothing=0.05: prevents overconfidence.
    """

    def __init__(self, gamma: float = 2.0, label_smoothing: float = 0.05):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        C    = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        p    = torch.exp(logp)

        # smooth one-hot
        with torch.no_grad():
            smooth = torch.full_like(logits, self.label_smoothing / C)
            smooth.scatter_(1, targets.unsqueeze(1),
                            1.0 - self.label_smoothing + self.label_smoothing / C)

        loss = -(((1.0 - p) ** self.gamma) * smooth * logp).sum(dim=1)
        return loss.mean()


criterion = FocalLoss(gamma=2.0, label_smoothing=0.05)
f1_metric = MulticlassF1Score(num_classes=NUM_LABELS, average='macro').to(DEVICE)
scaler    = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print('Focal Loss (gamma=2.0, label_smoothing=0.05)')
print('Metric : Macro F1')
print(f'AMP    : {DEVICE == "cuda"}')


## Cell 22 — Mixup Augmentation

Applied at the batch level — blends `input_values` tensors and their labels proportionally. Applied to 50% of batches.

In [ ]:
# ── CELL 22: Mixup helpers ─────────────────────────────
def mixup_batch(iv: torch.Tensor, labels: torch.Tensor, alpha: float = MIXUP_ALPHA):
    """Mix two shuffled versions of the batch.

    Returns: (mixed_iv, labels_a, labels_b, lambda)
    """
    lam  = float(np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    perm = torch.randperm(iv.size(0), device=iv.device)
    return lam * iv + (1 - lam) * iv[perm], labels, labels[perm], lam


def mixup_loss(crit, logits, y_a, y_b, lam):
    return lam * crit(logits, y_a) + (1 - lam) * crit(logits, y_b)


print(f'Mixup helpers ready! (alpha={MIXUP_ALPHA}, applied to {int(MIXUP_PROB*100)}% of batches)')


## Cell 23 — Training & Validation Functions

Adds mixed-precision (AMP), SpecAugment on-batch, and Mixup to the baseline training loop. All metric history is tracked for learning curve plots.

In [ ]:
# ── CELL 23: train_one_epoch + validate ───────────────
history = {'train_loss': [], 'train_f1': [], 'val_f1': []}
_global_step = 0


def train_one_epoch(model, loader, optimizer, scheduler, tag='train'):
    global _global_step
    model.train()
    f1_metric.reset()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc=f'Training [{tag}]')):
        iv     = batch['input_values'].to(DEVICE)   # (B, T, F)
        labels = batch['labels'].to(DEVICE)

        # SpecAugment (on-batch, before model forward)
        iv = spec_augment(iv)

        # Mixup (50% of batches)
        use_mix = (random.random() < MIXUP_PROB)
        if use_mix:
            iv, y_a, y_b, lam = mixup_batch(iv, labels)

        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(input_values=iv).logits
            if use_mix:
                loss = mixup_loss(criterion, logits, y_a, y_b, lam) / GRAD_ACCUM
            else:
                loss = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            _global_step += 1
            if USE_WANDB:
                wandb.log({'lr': scheduler.get_last_lr()[0]}, step=_global_step)

        total_loss += loss.item() * GRAD_ACCUM
        preds = torch.argmax(logits, dim=1)
        f1_metric.update(preds, labels if not use_mix else y_a)

    epoch_loss = total_loss / len(loader)
    epoch_f1   = f1_metric.compute().item()
    history['train_loss'].append(epoch_loss)
    history['train_f1'].append(epoch_f1)
    return epoch_loss, epoch_f1


def validate(model, loader):
    model.eval()
    f1_metric.reset()
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            iv     = batch['input_values'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                logits = model(input_values=iv).logits
            preds = torch.argmax(logits, dim=1)
            f1_metric.update(preds, labels)
    val_f1 = f1_metric.compute().item()
    history['val_f1'].append(val_f1)
    return val_f1


print('train_one_epoch + validate ready! (AMP + SpecAugment + Mixup)')


## Cell 24 — Phase 1: Classifier Head Warmup

All AST base model parameters frozen. Only the 10-class linear head trains. This is identical to the baseline but now using Focal Loss and AMP.

In [ ]:
# ── CELL 24: PHASE 1 — Head Only ──────────────────────
for param in model.audio_spectrogram_transformer.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable:,} trainable params (classifier head only)')

optimizer_p1   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_P1, weight_decay=WEIGHT_DECAY,
)
total_steps_p1 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(
    optimizer_p1,
    num_warmup_steps   = total_steps_p1 // 10,
    num_training_steps = total_steps_p1,
)

best_p1_f1 = 0.0
patience   = 0
print(f'\n{"="*55}')
print(f'PHASE 1 — HEAD WARMUP ({EPOCHS_P1} epochs, LR={LR_P1})')
print(f'{"="*55}\n')

for epoch in range(EPOCHS_P1):
    t_loss, t_f1 = train_one_epoch(model, train_loader, optimizer_p1, scheduler_p1, 'P1')
    v_f1 = validate(model, val_loader)
    print(f'Ep {epoch+1}/{EPOCHS_P1}  Loss={t_loss:.4f}  TrainF1={t_f1:.4f}  ValF1={v_f1:.4f}')
    if USE_WANDB: wandb.log({'p1/loss': t_loss, 'p1/train_f1': t_f1, 'p1/val_f1': v_f1})
    if v_f1 > best_p1_f1:
        best_p1_f1 = v_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, 'best_p1.pth'))
        patience = 0
        print(f'  Saved (val F1: {best_p1_f1:.4f})')
    else:
        patience += 1
        if patience >= PATIENCE_P1: print('  Early stop P1.'); break

print(f'\nPhase 1 done! Best val F1: {best_p1_f1:.4f}')


## Cell 25 — Phase 2: Top-6 Transformer Encoder Layers

Unfreeze only the last 6 of 12 transformer encoder layers plus the classifier head. The first 6 layers learn general audio features (edges, harmonics) that don't need changing — only top layers need to specialise for genre.

In [ ]:
# ── CELL 25: PHASE 2 — Top-6 Encoder Layers + Head ───
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'best_p1.pth'),
                                  map_location=DEVICE))

encoder_layers = model.audio_spectrogram_transformer.encoder.layer
n_total        = len(encoder_layers)       # 12
n_unfreeze     = 6
print(f'Total encoder layers: {n_total}  |  Unfreezing top {n_unfreeze}')

# Keep early layers frozen, unfreeze top-6
for i, layer in enumerate(encoder_layers):
    for p in layer.parameters():
        p.requires_grad = (i >= n_total - n_unfreeze)

# Always unfreeze head
for p in model.classifier.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 2: {trainable:,} trainable params')

top_layer_params  = [p for i, l in enumerate(encoder_layers)
                     for p in l.parameters()
                     if i >= n_total - n_unfreeze]
head_params       = list(model.classifier.parameters())

optimizer_p2 = torch.optim.AdamW([
    {'params': top_layer_params, 'lr': LR_P2_TOP},
    {'params': head_params,      'lr': LR_P2_HEAD},
], weight_decay=WEIGHT_DECAY)

total_steps_p2 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(
    optimizer_p2,
    num_warmup_steps   = total_steps_p2 // 10,
    num_training_steps = total_steps_p2,
)

best_p2_f1 = 0.0
patience   = 0
print(f'\n{"="*55}')
print(f'PHASE 2 — TOP-6 LAYERS ({EPOCHS_P2} epochs)')
print(f'Top LR={LR_P2_TOP}  Head LR={LR_P2_HEAD}')
print(f'{"="*55}\n')

for epoch in range(EPOCHS_P2):
    t_loss, t_f1 = train_one_epoch(model, train_loader, optimizer_p2, scheduler_p2, 'P2')
    v_f1 = validate(model, val_loader)
    print(f'Ep {epoch+1}/{EPOCHS_P2}  Loss={t_loss:.4f}  TrainF1={t_f1:.4f}  ValF1={v_f1:.4f}')
    if USE_WANDB: wandb.log({'p2/loss': t_loss, 'p2/train_f1': t_f1, 'p2/val_f1': v_f1})
    if v_f1 > best_p2_f1:
        best_p2_f1 = v_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, 'best_p2.pth'))
        patience = 0
        print(f'  Saved (val F1: {best_p2_f1:.4f})')
    else:
        patience += 1
        if patience >= PATIENCE_P2: print('  Early stop P2.'); break

print(f'\nPhase 2 done! Best val F1: {best_p2_f1:.4f}')


## Cell 26 — Phase 3: Full Fine-Tuning

Unfreeze all layers. Early encoder layers get `LR_P3=1e-5` (10× smaller than baseline `2e-5`). The classifier gets `LR_P3_HEAD=1e-4`. CosineAnnealingWarmRestarts gives a second chance to escape local minima.

In [ ]:
# ── CELL 26: PHASE 3 — Full Fine-Tune ─────────────────
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'best_p2.pth'),
                                  map_location=DEVICE))

for p in model.parameters():
    p.requires_grad = True

total = sum(p.numel() for p in model.parameters())
print(f'All {total:,} params unfrozen!')

optimizer_p3 = torch.optim.AdamW([
    {'params': model.audio_spectrogram_transformer.parameters(), 'lr': LR_P3},
    {'params': model.classifier.parameters(),                    'lr': LR_P3_HEAD},
], weight_decay=WEIGHT_DECAY)

total_steps_p3 = (len(train_loader) // GRAD_ACCUM) * EPOCHS_P3
scheduler_p3   = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_p3, T_0=max(1, total_steps_p3 // 2), T_mult=1, eta_min=1e-8,
)

best_p3_f1 = 0.0
patience   = 0
print(f'\n{"="*55}')
print(f'PHASE 3 — FULL FINE-TUNE ({EPOCHS_P3} epochs)')
print(f'Base LR={LR_P3}  |  Head LR={LR_P3_HEAD}')
print(f'{"="*55}\n')

for epoch in range(EPOCHS_P3):
    t_loss, t_f1 = train_one_epoch(model, train_loader, optimizer_p3, scheduler_p3, 'P3')
    v_f1 = validate(model, val_loader)
    print(f'Ep {epoch+1}/{EPOCHS_P3}  Loss={t_loss:.4f}  TrainF1={t_f1:.4f}  ValF1={v_f1:.4f}')
    if USE_WANDB: wandb.log({'p3/loss': t_loss, 'p3/train_f1': t_f1, 'p3/val_f1': v_f1})
    if v_f1 > best_p3_f1:
        best_p3_f1 = v_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_PATH, 'best_p3.pth'))
        patience = 0
        print(f'  Saved (val F1: {best_p3_f1:.4f})')
    else:
        patience += 1
        if patience >= PATIENCE_P3: print('  Early stop P3.'); break

print(f'\nPhase 3 done! Best val F1: {best_p3_f1:.4f}')


## Cell 27 — TTA Inference (N_TTA = 8)

Eight different random crops of the same audio, averaged via softmax. More crops → lower variance. This is the main reason Kaggle test score exceeds validation score.

In [ ]:
# ── CELL 27: TTA Inference ─────────────────────────────
def predict_with_tta(model: nn.Module, audio: np.ndarray, n_tta: int = N_TTA) -> int:
    """Run n_tta random crops through the AST and average softmax probabilities."""
    model.eval()
    all_probs = []
    for _ in range(n_tta):
        crop = crop_random(audio)
        crop = normalize(crop)
        iv   = feature_extractor(
            crop,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = 'pt',
        )['input_values'].to(DEVICE)

        with torch.no_grad():
            with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                logits = model(input_values=iv).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    avg = np.mean(all_probs, axis=0)
    return int(np.argmax(avg, axis=1)[0])


print(f'TTA ready! ({N_TTA} crops per prediction)')


## Cell 28 — Generate Submission

In [ ]:
# ── CELL 28: Generate Submission ──────────────────────
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'best_p3.pth'),
                                  map_location=DEVICE))
model.eval()
print(f'Best Phase-3 model loaded. Val F1: {best_p3_f1:.4f}')

test_df = pd.read_csv(os.path.join(BASE_PATH, 'test.csv'))
print(f'Test samples: {len(test_df)}')

all_ids, all_preds = [], []
t0 = time.time()

for idx in tqdm(range(len(test_df)), desc=f'Inference TTA×{N_TTA}'):
    row   = test_df.iloc[idx]
    path  = os.path.join(BASE_PATH, row['filename'])
    audio = load_audio(path)
    pred  = predict_with_tta(model, audio, n_tta=N_TTA)
    all_ids.append(row['id'])
    all_preds.append(id2label[pred])

elapsed    = time.time() - t0
submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
sub_path   = os.path.join(OUTPUT_PATH, 'submission.csv')
submission.to_csv(sub_path, index=False)

print(f'\nSubmission saved to {sub_path}')
print(f'Speed: {elapsed / len(test_df) * 1000:.1f} ms/sample')
print('\nGenre distribution:')
print(submission['genre'].value_counts())
print('\nFirst 5 rows:')
print(submission.head())


## Cell 29 — Analysis: Confusion Matrix · Per-Class F1 · Learning Curves

In [ ]:
# ── CELL 29: Analysis ─────────────────────────────────
model.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'best_p3.pth'),
                                  map_location=DEVICE))
model.eval()

all_p, all_t = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc='Analysis pass'):
        iv     = batch['input_values'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            logits = model(input_values=iv).logits
        all_p.extend(torch.argmax(logits, 1).cpu().numpy())
        all_t.extend(labels.cpu().numpy())

cm     = confusion_matrix(all_t, all_p)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

# --- Confusion matrices (count + row-normalised) ---
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=GENRES, yticklabels=GENRES, ax=axes[0])
axes[0].set_title('Confusion Matrix — Count', fontsize=13)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=GENRES, yticklabels=GENRES, ax=axes[1])
axes[1].set_title('Row-Normalised (%) = Recall per Class', fontsize=13)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout()
if USE_WANDB: wandb.log({'analysis/confusion_matrix': wandb.Image(fig)})
plt.show(); plt.close(fig)

# --- Per-class F1 bar chart ---
report     = classification_report(all_t, all_p, target_names=GENRES, output_dict=True)
per_cls_f1 = {g: report[g]['f1-score'] for g in GENRES}
fig2, ax2  = plt.subplots(figsize=(10, 4))
colors_    = ['#27ae60' if v >= 0.80 else '#e74c3c' for v in per_cls_f1.values()]
ax2.bar(per_cls_f1.keys(), per_cls_f1.values(), color=colors_, edgecolor='white')
ax2.axhline(0.80, color='black', linestyle='--', linewidth=1.5, label='0.80 target')
ax2.set_title('Per-Class F1 Score (green = meets target)', fontsize=13)
ax2.set_ylim(0, 1.05); ax2.legend()
plt.tight_layout()
if USE_WANDB: wandb.log({'analysis/per_class_f1': wandb.Image(fig2)})
plt.show(); plt.close(fig2)

# --- Learning curves ---
if history['train_loss']:
    n_ep = len(history['train_loss'])
    fig3, axes3 = plt.subplots(1, 2, figsize=(14, 4))
    axes3[0].plot(range(1, n_ep+1), history['train_loss'], 'b-o', markersize=4, label='Train Loss')
    axes3[0].set_title('Training Loss'); axes3[0].set_xlabel('Epoch'); axes3[0].legend()

    axes3[1].plot(range(1, n_ep+1), history['train_f1'], 'b-o', markersize=4, label='Train F1')
    axes3[1].plot(range(1, len(history['val_f1'])+1), history['val_f1'],
                  'r-o', markersize=4, label='Val F1')
    axes3[1].axhline(0.80, color='gray', linestyle='--', linewidth=1, label='Target 0.80')
    axes3[1].set_title('F1: Train vs Val')
    axes3[1].set_xlabel('Epoch'); axes3[1].legend()
    plt.tight_layout()
    if USE_WANDB: wandb.log({'analysis/learning_curves': wandb.Image(fig3)})
    plt.show(); plt.close(fig3)

print('\nClassification Report:')
print(classification_report(all_t, all_p, target_names=GENRES))


## Cell 30 — Final Summary

In [ ]:
# ── CELL 30: Summary ──────────────────────────────────
print('=' * 60)
print('TRAINING COMPLETE — Enhanced AST Fine-Tuning')
print('=' * 60)
print(f'  Phase 1 best val F1  : {best_p1_f1:.4f}  (head only)')
print(f'  Phase 2 best val F1  : {best_p2_f1:.4f}  (top-6 layers)')
print(f'  Phase 3 best val F1  : {best_p3_f1:.4f}  (full fine-tune)')
print(f'  TTA crops            : {N_TTA}')
print(f'  Duration             : {DURATION}s')
print(f'  Train size           : {TRAIN_SIZE}')
print(f'  Expected Kaggle F1   : ~{best_p3_f1 + 0.015:.4f}  (TTA test-time boost)')
print('=' * 60)
print()
print('Improvements over baseline AST_Improved_Training.ipynb:')
print('  1. 3-phase training (Head / Top-6 / Full) vs 2-phase')
print('  2. Pitch shift augmentation (+-3 semitones, 30% prob)')
print('  3. SpecAugment on log-mel tensors (freq + time masking)')
print('  4. Mixup at batch level (50% of batches, alpha=0.4)')
print('  5. Focal Loss gamma=2 vs CrossEntropy')
print('  6. Mixed-precision AMP throughout training')
print('  7. DURATION 25s vs 20s (+25% audio context)')
print('  8. TRAIN_SIZE 7000 vs 5000 (+40% diversity)')
print('  9. N_TTA 8 vs 5 (+60% inference crops)')
print(' 10. LR_P3 1e-5 vs 2e-5 (2x safer full fine-tune)')

if USE_WANDB:
    wandb.log({
        'summary/p1_f1': best_p1_f1,
        'summary/p2_f1': best_p2_f1,
        'summary/p3_f1': best_p3_f1,
    })
    wandb.finish()
